In [ ]:
# Imports
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Add ruthless core to path
ruthless_core_path = Path.cwd().parent / 'phase4' / '06_Ruthless_Core_Model'
if str(ruthless_core_path) not in sys.path:
    sys.path.insert(0, str(ruthless_core_path))

from ruthless_core import RuthlessCoreConfig, RuthlessCoreModel

print("Imports successful!")

## 1. Load Synthetic Real Data

We'll load a synthetic "real" cohesion trajectory with a visible third quarter dip.

In [ ]:
# Load real cohesion data
data_path = Path.cwd().parent / 'data' / 'example_real_cohesion.csv'
real_data = pd.read_csv(data_path)

print(f"Loaded {len(real_data)} days of real cohesion data")
print(f"Day range: {real_data['day'].min()} to {real_data['day'].max()}")
print(f"Cohesion range: {real_data['real_cohesion'].min():.3f} to {real_data['real_cohesion'].max():.3f}")

# Quick plot
plt.figure(figsize=(10, 4))
plt.plot(real_data['day'], real_data['real_cohesion'], 'o-', label='Real Cohesion', color='black')
plt.xlabel('Day')
plt.ylabel('Social Cohesion')
plt.title('Synthetic "Real" Cohesion Data')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 2. Run Baseline Simulation

Run the model with default parameters to see baseline behavior.

In [ ]:
# Create baseline configuration
config = RuthlessCoreConfig(
    mission_length_days=200,
    q_center=0.70,
    q_peak=0.5,
    q_width=0.15,
)

# Run model
model = RuthlessCoreModel(config)
output = model.run()

print(f"Simulation complete: {len(output)} days")
print(f"Cohesion range: {min(output.cohesion):.3f} to {max(output.cohesion):.3f}")

## 3. Compare Real vs Simulated

Plot real and simulated cohesion on the same axes.

In [ ]:
plt.figure(figsize=(12, 5))

# Plot real data
plt.plot(real_data['day'], real_data['real_cohesion'], 
         'o', markersize=4, label='Real Cohesion', color='black', alpha=0.6)

# Plot simulated cohesion
plt.plot(output.days, output.cohesion, 
         '-', linewidth=2, label='Simulated Cohesion', color='blue')

plt.xlabel('Day')
plt.ylabel('Social Cohesion')
plt.title('Real vs Simulated Cohesion')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Compute Error Metrics

Calculate mean squared error and other fit statistics.

In [ ]:
# Interpolate simulated cohesion at real data days
simulated_at_real_days = np.interp(real_data['day'], output.days, output.cohesion)

# Compute metrics
errors = real_data['real_cohesion'].values - simulated_at_real_days
mse = np.mean(errors ** 2)
rmse = np.sqrt(mse)
mae = np.mean(np.abs(errors))
max_error = np.max(np.abs(errors))

print("=== Error Metrics ===")
print(f"Mean Squared Error (MSE):  {mse:.6f}")
print(f"Root Mean Squared Error:   {rmse:.6f}")
print(f"Mean Absolute Error:       {mae:.6f}")
print(f"Max Absolute Error:        {max_error:.6f}")

# Plot error over time
plt.figure(figsize=(12, 4))
plt.plot(real_data['day'], errors, 'o-', color='red', alpha=0.6)
plt.axhline(0, color='black', linestyle='--', linewidth=1)
plt.xlabel('Day')
plt.ylabel('Error (Real - Simulated)')
plt.title('Prediction Error Over Time')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Visualize All State Variables

Plot monotony, strain, TQ pressure, and cohesion together.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Monotony
axes[0, 0].plot(output.days, output.monotony, color='orange', linewidth=2)
axes[0, 0].set_title('Monotony M(t)')
axes[0, 0].set_xlabel('Day')
axes[0, 0].set_ylabel('Monotony')
axes[0, 0].grid(True, alpha=0.3)

# Strain
axes[0, 1].plot(output.days, output.strain, color='red', linewidth=2)
axes[0, 1].set_title('Psychological Strain S(t)')
axes[0, 1].set_xlabel('Day')
axes[0, 1].set_ylabel('Strain')
axes[0, 1].grid(True, alpha=0.3)

# TQ Pressure
axes[1, 0].plot(output.days, output.tq_pressure, color='purple', linewidth=2)
axes[1, 0].set_title('Third Quarter Pressure Q(t)')
axes[1, 0].set_xlabel('Day')
axes[1, 0].set_ylabel('TQ Pressure')
axes[1, 0].grid(True, alpha=0.3)

# Cohesion
axes[1, 1].plot(output.days, output.cohesion, color='blue', linewidth=2)
axes[1, 1].plot(real_data['day'], real_data['real_cohesion'], 
                'o', markersize=3, color='black', alpha=0.4, label='Real')
axes[1, 1].set_title('Social Cohesion C(t)')
axes[1, 1].set_xlabel('Day')
axes[1, 1].set_ylabel('Cohesion')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Parameter Tuning Cell

Adjust parameters here and re-run to see how the fit changes.

**Key parameters to adjust:**
- `q_center`: Position of third quarter peak (0.66-0.75)
- `q_peak`: Amplitude of TQ pressure (0.3-0.8)
- `q_width`: Width of TQ effect (0.10-0.20)
- `c_strain`: Cohesion erosion from strain
- `c_q`: Cohesion erosion from TQ pressure

In [ ]:
# ============================================================
# ADJUST PARAMETERS HERE
# ============================================================

config_tuned = RuthlessCoreConfig(
    mission_length_days=200,
    
    # Third quarter parameters - TUNE THESE
    q_center=0.68,      # Try values between 0.66 and 0.75
    q_peak=0.6,         # Try values between 0.3 and 0.8
    q_width=0.12,       # Try values between 0.10 and 0.20
    
    # Cohesion parameters - TUNE THESE
    c_strain=0.018,     # Increase to make strain erode cohesion more
    c_q=0.025,          # Increase to make TQ pressure erode cohesion more
    c_shared_success=0.05,
    
    # Strain parameters
    s_workload=0.05,
    s_mono=0.02,
    s_recovery=0.03,
    
    # Monotony parameters
    m_base=0.01,
    m_novelty=0.2,
)

# Run tuned model
model_tuned = RuthlessCoreModel(config_tuned)
output_tuned = model_tuned.run()

# Compute new error
simulated_tuned = np.interp(real_data['day'], output_tuned.days, output_tuned.cohesion)
errors_tuned = real_data['real_cohesion'].values - simulated_tuned
mse_tuned = np.mean(errors_tuned ** 2)
rmse_tuned = np.sqrt(mse_tuned)

print(f"Tuned Model RMSE: {rmse_tuned:.6f}")
print(f"Improvement: {(rmse - rmse_tuned):.6f}")

# Plot comparison
plt.figure(figsize=(12, 5))
plt.plot(real_data['day'], real_data['real_cohesion'], 
         'o', markersize=4, label='Real Cohesion', color='black', alpha=0.6)
plt.plot(output.days, output.cohesion, 
         '--', linewidth=2, label='Baseline Sim', color='blue', alpha=0.5)
plt.plot(output_tuned.days, output_tuned.cohesion, 
         '-', linewidth=2, label='Tuned Sim', color='green')
plt.xlabel('Day')
plt.ylabel('Social Cohesion')
plt.title('Calibration: Real vs Baseline vs Tuned')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Automated Parameter Fitting

Uses Nelder-Mead optimization to find the best-fit values for the 5 key parameters:

| Parameter | Role | Search range |
|-----------|------|--------------|
| `q_center` | Where the TQP peak lands (mission fraction) | 0.50 – 0.85 |
| `q_peak` | Amplitude of TQP pressure | 0.15 – 1.20 |
| `q_width` | Width of the Gaussian TQP hump | 0.05 – 0.30 |
| `c_strain` | Cohesion erosion rate from strain | 0.002 – 0.060 |
| `c_q` | Cohesion erosion rate from TQP pressure | 0.005 – 0.100 |

Uses **scipy** Nelder-Mead if installed (`pip install scipy`), otherwise falls back to the built-in pure-Python implementation.

To calibrate against real data, replace `data_path` with your CSV (same format: `day,real_cohesion`).

In [ ]:
import sys
from pathlib import Path

# Add calibration module to path
calibration_path = Path.cwd().parent / 'calibration'
if str(calibration_path) not in sys.path:
    sys.path.insert(0, str(calibration_path))

from fit_cohesion import calibrate

# ─────────────────────────────────────────────────────────────
# Run the optimizer
# Replace data_path to point at real MARS-500 / analog data
# ─────────────────────────────────────────────────────────────
fit_result = calibrate(
    data_path=str(Path.cwd().parent / 'data' / 'example_real_cohesion.csv'),
    mission_length=200,
    output_path=str(Path.cwd().parent / 'data' / 'fitted_config.json'),
    use_scipy=True,   # set False to force pure-Python fallback
    verbose=True,
)

print("\nOptimized parameters:")
for k, v in fit_result['optimized_params'].items():
    print(f"  {k:12s} = {v:.5f}")

print(f"\nFit quality:")
m = fit_result['fit_metadata']
print(f"  RMSE      = {m['rmse']:.6f}")
print(f"  MAE       = {m['mae']:.6f}")
print(f"  max_error = {m['max_error']:.6f}")
print(f"  optimizer = {m['optimizer']}")

In [ ]:
# Plot: Real vs Baseline vs Fitted
fitted_cfg = fit_result['full_config']

config_fitted = RuthlessCoreConfig(
    mission_length_days=fitted_cfg['mission_length_days'],
    q_center=fitted_cfg['q_center'],
    q_peak=fitted_cfg['q_peak'],
    q_width=fitted_cfg['q_width'],
    c_strain=fitted_cfg['c_strain'],
    c_q=fitted_cfg['c_q'],
)
output_fitted = RuthlessCoreModel(config_fitted).run()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: cohesion comparison ---
ax = axes[0]
ax.plot(real_data['day'], real_data['real_cohesion'],
        'o', markersize=4, color='black', alpha=0.7, label='Real data', zorder=3)
ax.plot(output.days, output.cohesion,
        '--', linewidth=2, color='steelblue', alpha=0.6, label='Baseline (defaults)')
ax.plot(output_fitted.days, output_fitted.cohesion,
        '-', linewidth=2.5, color='forestgreen', label='Fitted')
ax.set_xlabel('Day')
ax.set_ylabel('Social Cohesion')
ax.set_title('Calibration Result — Cohesion Trajectory')
ax.legend()
ax.grid(True, alpha=0.3)

# Annotate RMSE
rmse_baseline = np.sqrt(np.mean(
    (real_data['real_cohesion'].values - np.interp(real_data['day'], output.days, output.cohesion)) ** 2
))
rmse_fitted = fit_result['fit_metadata']['rmse']
ax.text(0.02, 0.06,
        f"Baseline RMSE: {rmse_baseline:.4f}\nFitted RMSE:   {rmse_fitted:.4f}",
        transform=ax.transAxes, fontsize=9,
        verticalalignment='bottom', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# --- Right: residuals ---
ax2 = axes[1]
fitted_at_real = np.interp(real_data['day'], output_fitted.days, output_fitted.cohesion)
residuals = real_data['real_cohesion'].values - fitted_at_real
ax2.bar(real_data['day'], residuals, width=3, color=['tomato' if r < 0 else 'steelblue' for r in residuals], alpha=0.7)
ax2.axhline(0, color='black', linewidth=1)
ax2.set_xlabel('Day')
ax2.set_ylabel('Residual (Real − Fitted)')
ax2.set_title('Fitted Model Residuals')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nOptimized config saved to: {Path.cwd().parent / 'data' / 'fitted_config.json'}")

## 8. Summary

Use this notebook to:
1. Visually compare model output against real data
2. Iteratively tune parameters manually in section 6
3. Run the automated optimizer in section 7 for systematic fitting
4. Monitor error metrics (RMSE, MAE) to track improvement
5. Validate that the model can reproduce the third quarter dip

**Interpreting the fit:**
- If optimized params hit their bounds, the data may lack a clear TQP signature (common in synthetic or pre-processed data)
- With real MARS-500 / HI-SEAS data, `q_center` and `q_width` will lock onto the actual dip position
- RMSE < 0.05 is acceptable for long-duration cohesion data; < 0.02 is strong

**Using fitted parameters in production:**
```python
import json
from ruthless_core import RuthlessCoreConfig, RuthlessCoreModel

with open('../data/fitted_config.json') as f:
    cfg = json.load(f)['full_config']

config = RuthlessCoreConfig(**{k: v for k, v in cfg.items()
                               if not k.endswith('_schedule') and k != 'mission_length_days'},
                            mission_length_days=cfg['mission_length_days'])
output = RuthlessCoreModel(config).run()
```

**Replacing the synthetic data with real data:**
- Source: MARS-500 (520 days), HI-SEAS (180 days), Concordia (12 months)
- Format: CSV with `day` (integer) and `real_cohesion` (float 0–1, normalized)
- Place in `data/` and update `data_path` in section 7